In [1]:
!pip install tqdm
!pip install ipywidgets
!pip show jupyterlab

Name: jupyterlab
Version: 4.2.0
Summary: JupyterLab computational environment
Home-page: 
Author: 
Author-email: Jupyter Development Team <jupyter@googlegroups.com>
License: Copyright (c) 2015-2022 Project Jupyter Contributors
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUT

In [2]:
from datetime import datetime, timedelta
import json
from tqdm import tqdm
import re

def find_dates(text):
    date_patterns = {
        r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{1,6}': "%Y-%m-%d %H:%M:%S.%f",
        r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}': "%Y-%m-%d %H:%M:%S",
        r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{1,6}Z': "%Y-%m-%dT%H:%M:%S.%fZ",
        r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z': "%Y-%m-%dT%H:%M:%SZ"
    }
    for pattern, date_format in date_patterns.items():
        match = re.search(pattern, text)
        if match:
            date_str = match.group()
            return datetime.strptime(date_str, date_format), date_format
    return None, None

def adjust_and_combine_logs(file_paths, output_file):
    all_logs = []
    most_recent_time = None
    for file_path in file_paths:
        with open(file_path, 'r') as file:
            for line in tqdm(file, desc="Processing logs"):
                if line.strip():
                    log = json.loads(line)
                    all_logs.append(log)
                    for key, value in log.items():
                        if isinstance(value, str):
                            date, fmt = find_dates(value)
                            if date and (not most_recent_time or date > most_recent_time):
                                most_recent_time = date

    reference_time = datetime.now() - timedelta(minutes=15)
    if most_recent_time:
        time_diff = reference_time - most_recent_time
        for log in tqdm(all_logs, desc="Adjusting dates"):
            for key, value in log.items():
                if isinstance(value, str):
                    date, fmt = find_dates(value)
                    if date:
                        new_date = date + time_diff
                        formatted_date = new_date.strftime(fmt)
                        log[key] = re.sub(r'\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(.\d{1,6})?Z?', formatted_date, value)

    with open(output_file, 'w') as file:
        for log in tqdm(all_logs, desc="Writing logs"):
            file.write(json.dumps(log, separators=(',', ':')) + '\n')

file_paths = ['./apt29/apt29_evals_day1_manual_2020-05-01225525.json', './apt29/apt29_evals_day2_manual_2020-05-02035409.json']
output_file = './apt29/apt29.json'
adjust_and_combine_logs(file_paths, output_file)


Processing logs: 196081it [00:28, 6890.12it/s]
Processing logs: 587286it [02:07, 4619.22it/s]
Writing logs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 783367/783367 [00:18<00:00, 43187.22it/s]


In [3]:
# Carregar e exibir as primeiras entradas do arquivo combinado
with open('./apt29/apt29.json', 'r') as file:
    for _ in range(10):
        print(file.readline())  # Ler e imprimir as primeiras 10 linhas

{"EventTime":"2024-05-20 05:11:16","port":60737,"Message":"Process accessed:\r\nRuleName: -\r\nUtcTime: 2024-05-20 09:11:16.895846\r\nSourceProcessGUID: {6bbf237a-cafb-5eac-1000-000000000400}\r\nSourceProcessId: 900\r\nSourceThreadId: 504\r\nSourceImage: C:\\windows\\system32\\svchost.exe\r\nTargetProcessGUID: {6bbf237a-cb97-5eac-6202-000000000400}\r\nTargetProcessId: 2092\r\nTargetImage: C:\\windows\\System32\\svchost.exe\r\nGrantedAccess: 0x1000\r\nCallTrace: C:\\windows\\SYSTEM32\\ntdll.dll+9c584|C:\\windows\\SYSTEM32\\psmserviceexthost.dll+222a3|C:\\windows\\SYSTEM32\\psmserviceexthost.dll+1a172|C:\\windows\\SYSTEM32\\psmserviceexthost.dll+19e3b|C:\\windows\\SYSTEM32\\psmserviceexthost.dll+19318|C:\\windows\\SYSTEM32\\ntdll.dll+3089d|C:\\windows\\SYSTEM32\\ntdll.dll+34634|C:\\windows\\System32\\KERNEL32.DLL+17bd4|C:\\windows\\SYSTEM32\\ntdll.dll+6ced1","SourceThreadId":"504","EventID":10,"TargetProcessId":"2092","SourceModuleName":"eventlog","tags":["mordorDataset"],"@version":"1",

In [4]:
# Abre o arquivo para leitura
with open('./apt29/apt29.json', 'r') as file:
    lines = file.readlines()  # Lê todas as linhas do arquivo

# Imprime as últimas 10 linhas
for line in lines[-10:]:
    print(line)

{"Message":"Process accessed:\r\nRuleName: -\r\nUtcTime: 2024-05-20 14:45:15.364846\r\nSourceProcessGUID: {fe9afb36-194b-5ead-1200-000000000500}\r\nSourceProcessId: 372\r\nSourceThreadId: 1692\r\nSourceImage: C:\\windows\\system32\\svchost.exe\r\nTargetProcessGUID: {fe9afb36-194a-5ead-0c00-000000000500}\r\nTargetProcessId: 736\r\nTargetImage: C:\\windows\\system32\\lsass.exe\r\nGrantedAccess: 0x1000\r\nCallTrace: C:\\windows\\SYSTEM32\\ntdll.dll+9c584|C:\\windows\\System32\\KERNELBASE.dll+6a8b5|c:\\windows\\system32\\lsm.dll+ff97|C:\\windows\\System32\\RPCRT4.dll+76953|C:\\windows\\System32\\RPCRT4.dll+da036|C:\\windows\\System32\\RPCRT4.dll+37a4c|C:\\windows\\System32\\RPCRT4.dll+548c8|C:\\windows\\System32\\RPCRT4.dll+2c921|C:\\windows\\System32\\RPCRT4.dll+2c1db|C:\\windows\\System32\\RPCRT4.dll+1a86f|C:\\windows\\System32\\RPCRT4.dll+19d1a|C:\\windows\\System32\\RPCRT4.dll+19301|C:\\windows\\System32\\RPCRT4.dll+18d6e|C:\\windows\\System32\\RPCRT4.dll+169a5|C:\\windows\\SYSTEM32\\n